In [1]:
!pip install fpdf

  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=f25f00da1301592bd57e549b30977a0e055857e348b9982380f86d38addc7325
  Stored in directory: /root/.cache/pip/wheels/6e/62/11/dc73d78e40a218ad52e7451f30166e94491be013a7850b5d75
Successfully built fpdf


In [14]:
import json
import re
import os
from fpdf import FPDF
from datetime import datetime

INPUT_FILE = '/content/clinical_trials.jsonl'
OUTPUT_DIR = 'CSSR_PDFs'

# Create output directory
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def extract_semantic_sections(semantic_text):
    sections = {
        'Official Title': 'Not Provided', 'Summary': 'Not Provided',
        'Study Type': 'Not Provided', 'Inclusion Criteria': 'Not Provided',
        'Exclusion Criteria': 'Not Provided', 'Primary Outcomes': 'Not Provided',
        'Secondary Outcomes': 'Not Provided'
    }

    off_title = re.search(r'Official Title:\s*(.*?)\n\n', semantic_text, re.DOTALL)
    if off_title: sections['Official Title'] = off_title.group(1).strip()

    summary = re.search(r'Summary:\s*(.*?)\n\n', semantic_text, re.DOTALL)
    if summary: sections['Summary'] = summary.group(1).strip()

    study_design = re.search(r'Study Design:\s*(.*?)\n\n', semantic_text, re.DOTALL)
    if study_design: sections['Study Type'] = study_design.group(1).strip()

    eligibility = re.search(r'Eligibility Criteria:(.*?)(?=\n\nPrimary Outcomes:|$)', semantic_text, re.DOTALL)
    if eligibility:
        elig_text = eligibility.group(1)
        incl = re.search(r'Inclusion Criteria:\s*(.*?)(?=\n\nExclusion Criteria:|$)', elig_text, re.DOTALL)
        excl = re.search(r'Exclusion Criteria:\s*(.*?)$', elig_text, re.DOTALL)
        if incl: sections['Inclusion Criteria'] = incl.group(1).strip()
        if excl: sections['Exclusion Criteria'] = excl.group(1).strip()

    prim_out = re.search(r'Primary Outcomes:\s*(.*?)(?=\n\nSecondary Outcomes:|$)', semantic_text, re.DOTALL)
    if prim_out: sections['Primary Outcomes'] = prim_out.group(1).strip()

    sec_out = re.search(r'Secondary Outcomes:\s*(.*?)$', semantic_text, re.DOTALL)
    if sec_out: sections['Secondary Outcomes'] = sec_out.group(1).strip()

    return sections

class CSSR_PDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 14)
        self.cell(0, 10, 'Clinical Study Summary Report (CSSR)', 0, 1, 'C')
        self.set_font('Arial', '', 10)
        self.cell(0, 6, 'Document Status: Final | Version: 1.0 | Date: 01-APR-2026', 0, 1, 'C')
        self.ln(5)

    def chapter_title(self, title):
        self.set_font('Arial', 'B', 12)
        self.set_fill_color(200, 220, 255)
        self.cell(0, 8, title, 0, 1, 'L', 1)
        self.ln(2)

    def chapter_body(self, text):
        self.set_font('Arial', '', 10)
        clean_text = str(text).encode('latin-1', 'replace').decode('latin-1')
        self.multi_cell(0, 6, clean_text)
        self.ln(2)

    def key_value(self, key, value):
        self.set_font('Arial', 'B', 10)
        self.cell(50, 6, str(key) + ':', 0, 0)
        self.set_font('Arial', '', 10)
        clean_val = str(value).encode('latin-1', 'replace').decode('latin-1')
        self.multi_cell(0, 6, clean_val)
        self.ln(1)

def generate_pdf(trial):
    pdf = CSSR_PDF()
    pdf.add_page()
    sem_data = extract_semantic_sections(trial.get('semantic_text', ''))

    pdf.chapter_title('1. Administrative & Study Identification')
    pdf.key_value('Full Study Title', sem_data['Official Title'])
    pdf.key_value('Short Title', trial.get('StudyTitle', ''))
    pdf.key_value('Protocol Number', trial.get('PostingID', ''))
    pdf.key_value('NCT Number', trial.get('NCT_Number', ''))
    pdf.key_value('Sponsor Name', trial.get('Sponsor', ''))
    pdf.key_value('Investigational Product', trial.get('drug_preferred_name', 'Not Provided'))
    pdf.key_value('Phase', trial.get('Phase', ''))
    pdf.key_value('Medical Monitor', '[Not Provided in Posting]')
    pdf.ln(5)

    pdf.chapter_title('2. Study Synopsis & Rationale')
    pdf.set_font('Arial', 'B', 10)
    pdf.cell(0, 6, '2.1 Study Objective', 0, 1)
    pdf.key_value('Primary Objective', sem_data['Primary Outcomes'])
    pdf.key_value('Secondary Obj.', sem_data['Secondary Outcomes'])
    pdf.ln(3)
    pdf.set_font('Arial', 'B', 10)
    pdf.cell(0, 6, '2.2 Background & Rationale', 0, 1)
    pdf.chapter_body(sem_data['Summary'])

    pdf.chapter_title('3. Study Design & Methodology')
    pdf.key_value('Design Framework', sem_data['Study Type'])
    pdf.key_value('Target N', trial.get('Targeted_Enrollment', '[Not Provided]'))
    pdf.ln(5)

    pdf.chapter_title('4. Subject Selection (Eligibility Criteria)')
    pdf.set_font('Arial', 'B', 10)
    pdf.cell(0, 6, '4.1 Inclusion Criteria', 0, 1)
    pdf.chapter_body(sem_data['Inclusion Criteria'])
    pdf.set_font('Arial', 'B', 10)
    pdf.cell(0, 6, '4.2 Exclusion Criteria', 0, 1)
    pdf.chapter_body(sem_data['Exclusion Criteria'])

    pdf.chapter_title('11. Study Identifiers & Governance')
    pdf.key_value('Primary ID', trial.get('PostingID', ''))
    pdf.key_value('Registry Number', trial.get('NCT_Number', ''))
    pdf.key_value('Sponsor', trial.get('Sponsor', ''))
    pdf.key_value('Study Title', trial.get('StudyTitle', ''))
    pdf.ln(5)

    pdf.chapter_title('15. Sign-off & Approval')
    pdf.chapter_body('Principal Investigator: ______________________ Date: ________\n'
                     'Clinical Operations Lead: ____________________ Date: ________\n'
                     'Lead Statistician: ___________________________ Date: ________')

    filename = f"{OUTPUT_DIR}/CSSR_{trial.get('PostingID', 'unknown')}.pdf"
    pdf.output(filename)

# Run the generator
count = 0
line_num = 0
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line_num += 1
        if line.strip():
            try:
                trial_data = json.loads(line)
                generate_pdf(trial_data)
                count += 1
            except json.JSONDecodeError as e:
                print(f"Skipping malformed JSON line {line_num}: {e} - Content: {line.strip()[:100]}...")

print(f"Successfully generated {count} PDF files in the '{OUTPUT_DIR}' folder!")

# Zip the folder so you can download them all at once
import shutil
shutil.make_archive('CSSR_PDFs', 'zip', OUTPUT_DIR)
print("Created CSSR_PDFs.zip. You can now download it from the Files pane.")

Successfully generated 174 PDF files in the 'CSSR_PDFs' folder!
Created CSSR_PDFs.zip. You can now download it from the Files pane.


In [13]:
!rm -rf /content/CSSR_PDFs/

In [7]:
!touch /content/clinical_trials.jsonl